# 01 · Prepare the datasets

**Settings:** Accelerator *None*, Internet off.
**Inputs:** `bbd-code`, a TON_IoT dataset (the `train_test_network` CSV), and a CICIoT2023 dataset (the CSV part files).

The notebook finds both datasets by their **columns**, not their folder names, so any Kaggle copy with the
original schema works. It drops identifiers that leak the label (timestamps, IPs, ports), removes rows whose
features appear with conflicting labels and exact duplicates, fixes balanced test (300/class) and validation
(100/class) sets, and saves three prepared datasets: `toniot8` (main, 8 classes: without MITM, whose
labels are unreliable, and ransomware, of which only ~1% of rows survive de-duplication), `toniot10` (all classes,
sensitivity check) and `ciciot8` (8 categories). CICIoT2023 is streamed file by file and capped at 60,000
rows per class, sampled uniformly over all files so the rare Web and BruteForce rows are all kept.
Expect about 20–40 minutes for CICIoT2023.

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
from bbd import data, config
src = data.find_sources(config.INPUT_ROOT)
for k, v in src.items():
    print(f"{k}: {len(v)} file(s)", *[f"  {p}" for p in v[:3]], "  ..." if len(v) > 3 else "", sep="\n")
assert src["toniot"], "No TON_IoT file found (needs columns type, src_bytes, conn_state). See KAGGLE_INSTRUCTIONS.md."
assert src["ciciot"], "No CICIoT2023 files found (needs columns label, flow_duration, Protocol Type). See KAGGLE_INSTRUCTIONS.md."

In [ ]:
import time
t = time.time(); ton = data.prepare_toniot(src["toniot"]); print(f"TON_IoT done in {time.time()-t:.0f}s")
t = time.time(); cic = data.prepare_ciciot(src["ciciot"], cap=60_000); print(f"CICIoT2023 done in {time.time()-t:.0f}s")
prepared = {**ton, **cic}
for p in prepared.values():
    data.save_prepared(p)
print("saved to", config.PREPARED_DIR)

In [ ]:
import pandas as pd
rows = []
for name, p in prepared.items():
    for i, c in enumerate(p.class_names):
        rows.append(dict(dataset=name, rank=i, cls=c, raw_rows=p.meta["raw_counts"].get(c, 0),
                         survival=round(p.meta["survival"][c], 3), natural_est=int(p.natural_counts[i]),
                         pool=int(p.pool_counts()[i]), val=int(((p.y == i) & (p.split == 1)).sum()),
                         test=int(((p.y == i) & (p.split == 2)).sum())))
summary = pd.DataFrame(rows)
summary.to_csv(config.PREPARED_DIR / "dataset_summary.csv", index=False)
display(summary)
for name, p in prepared.items():
    print(name, "features:", p.d, "| cleaning:", p.meta["dedup"], "| natural rho ~ %.0f" % (p.natural_counts.max() / p.natural_counts.min()))

In [ ]:
from bbd import figures
from IPython.display import Image, display
figures.fig_datasets([prepared["toniot8"], prepared["ciciot8"]], config.FIG_DIR)
display(Image(str(config.FIG_DIR / "fig_datasets.png"), width=900))